# ⚙️ Notebook 02 : Prétraitement Rigoureux & Prévention du Data Leakage

## 🎯 Objectif du Notebook
Le prétraitement des données (*Data Preprocessing*) est une étape critique et sensible. Un nettoyage mal conçu peut invalider la totalité de l'évaluation clinique d'un modèle d'Intelligence Artificielle en introduisant une fuite d'information (*Data Leakage*). Ce notebook met en œuvre un pipeline structuré pour :
1. **Séparer les données en amont** : procéder au découpage stratifié Entraînement/Test (*Train/Test Split*) avant toute transformation, afin d'isoler hermétiquement l'échantillon de test.
2. **Corriger les zéros masqués par imputation robuste** : remplacer les valeurs cliniques absurdes (`0` dans le glucose, l'insuline, l'IMC, etc.) par la médiane calculée exclusivement sur l'échantillon d'apprentissage.
3. **Harmoniser les échelles cliniques (*Feature Scaling*)** : standardiser les variables continues afin de garantir la convergence et l'équité des modèles mathématiques paramétriques.

---

## 1. Initialisation de l'Environnement et Chargement du Dataset


In [1]:
import sys
import os

import joblib
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_loader import load_raw_data
from src.preprocessing import split_data, impute_missing_values, scale_features, COLS_TO_IMPUTE

In [2]:
df = load_raw_data()
print(f"Dimensions du dataset : {df.shape}")
df.head()

Dimensions du dataset : (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Découpage Stratifié Entraînement / Test (*Train/Test Split*)

Avant de procéder à la moindre transformation ou correction des données, nous divisons le jeu de données en deux sous-ensembles distincts :
- **Échantillon d'apprentissage (`X_train`, 80 %)** : sert à entraîner les algorithmes et à estimer les paramètres statistiques de nettoyage (ex: la médiane d'imputation et l'écart-type de standardisation).
- **Échantillon d'évaluation (`X_test`, 20 %)** : strictement réservé à l'évaluation finale du modèle, agissant comme de nouvelles données cliniques réelles en milieu hospitalier.

> [!IMPORTANT]
> **Pourquoi le découpage doit-il précéder le nettoyage ? (Prévention du *Data Leakage*)**  
> Le *Data Leakage* (fuite de données) survient lorsque des informations issues du jeu de test influencent, même indirectement, le nettoyage ou l'apprentissage du modèle.  
> 
> *Exemple de faute méthodologique fréquente :* Si l'on calcule la médiane du glucose sur la totalité des 768 patientes avant le découpage, cette médiane intègre les informations des patientes de l'échantillon de test. Le modèle bénéficie alors indûment d'une connaissance préalable du jeu d'évaluation, ce qui entraîne des **scores de performance artificiellement gonflés et trompeurs en production médicale**.  
> 
> **La règle d'or en data science médicale :** tous les paramètres statistiques de transformation doivent être appris **strictement sur `X_train`**, puis appliqués (*transform*) à `X_test`.

De plus, nous appliquons une **stratification clinique (`stratify=y`)** lors du découpage afin de maintenir rigoureusement la même proportion de patientes diabétiques dans les deux ensembles (`~35 %`).


In [3]:
X_train, X_test, y_train, y_test = split_data(df)

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"Proportion Outcome=1 dans train : {y_train.mean():.2%}")
print(f"Proportion Outcome=1 dans test : {y_test.mean():.2%}")

X_train : (614, 8) | X_test : (154, 8)
Proportion Outcome=1 dans train : 34.85%
Proportion Outcome=1 dans test : 35.06%


## 3. Imputation Robuste des Zéros Masqués (*Missing Value Imputation*)

Lors de l'Analyse Exploratoire (`01_eda.ipynb`), nous avons diagnostiqué la présence de zéros biologiquement impossibles au sein de cinq variables vitales : `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` et `BMI`.

Notre fonction `impute_missing_values()` réalise deux opérations successives :
1. **Conversion explicite** : les chiffres `0` de ces cinq colonnes sont convertis en valeurs manquantes standardisées (`np.nan`).
2. **Imputation par la médiane du Train** : chaque `NaN` est remplacé par la **médiane** de la variable correspondante, calculée **strictement sur `X_train`**.

> [!NOTE]
> **Pourquoi choisir la Médiane plutôt que la Moyenne ?**  
> Comme observé sur nos histogrammes, les paramètres cliniques comme l'insuline présentent une forte asymétrie vers la droite avec de nombreuses valeurs extrêmes élevées (*outliers*). La médiane, étant un indicateur d'ordre de grandeur non paramétrique, est insensible aux valeurs extrêmes, garantissant une imputation clinique réaliste et stable.


In [4]:
X_train_clean, X_test_clean = impute_missing_values(X_train, X_test)

cols = COLS_TO_IMPUTE

print("Zéros restants après imputation :")
for c in cols:
    zeros_avant = (X_train[c] == 0).sum()
    zeros_apres = (X_train_clean[c] == 0).sum()
    print(f" {c:25s} : {zeros_avant} zéros → {zeros_apres} zéros")

Zéros restants après imputation :
 Glucose                   : 4 zéros → 0 zéros
 BloodPressure             : 23 zéros → 0 zéros
 SkinThickness             : 175 zéros → 0 zéros
 Insulin                   : 290 zéros → 0 zéros
 BMI                       : 9 zéros → 0 zéros


## 4. Standardisation des Échelles Cliniques (*Feature Scaling*)

Les variables cliniques brutes évoluent sur des échelles numériques hétérogènes :
- Le `DiabetesPedigreeFunction` varie entre `0.07` et `2.42`.
- L'`Insulin` peut atteindre plus de `800.0`.

Si les variables sont conservées en l'état, les modèles mathématiques basés sur le calcul de distance ou de gradient (comme la **Régression Logistique**) accorderont un poids excessif et artificiel aux variables possédant des valeurs numériques élevées, au détriment des variables sur de petites échelles.

Nous appliquons une **Standardisation (`StandardScaler`)** qui transforme chaque variable continue $x$ selon la formule z-score :
$$x_{	ext{scaled}} = rac{x - \mu}{\sigma}$$
où $\mu$ représente la moyenne et $\sigma$ l'écart-type de la variable, **estimés exclusivement sur `X_train_clean`**. Après cette transformation, chaque variable possède une moyenne nulle ($\mu = 0$) et un écart-type unitaire ($\sigma = 1$).


In [5]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train_clean, X_test_clean)

print("Moyennes par colonne (doit être ≈ 0) :")
print(X_train_scaled.mean(axis=0).round(4))

print("\nÉcarts-types par colonne (doit être ≈ 1) :")
print(X_train_scaled.std(axis=0, ddof=0).round(4))

Moyennes par colonne (doit être ≈ 0) :
Pregnancies                -0.0
Glucose                    -0.0
BloodPressure               0.0
SkinThickness              -0.0
Insulin                    -0.0
BMI                         0.0
DiabetesPedigreeFunction   -0.0
Age                        -0.0
dtype: float64

Écarts-types par colonne (doit être ≈ 1) :
Pregnancies                 1.0
Glucose                     1.0
BloodPressure               1.0
SkinThickness               1.0
Insulin                     1.0
BMI                         1.0
DiabetesPedigreeFunction    1.0
Age                         1.0
dtype: float64


## 5. Synthèse et Export des Matrices Nettoyées

Le pipeline de prétraitement est désormais achevé. Nous disposons de matrices de données parfaitement conformes aux exigences cliniques et algorithmiques :
- **Zéro fuite d'information (*No Data Leakage*)** : `X_test_scaled` a été traité comme un flux de données externe et hermétique.
- **Intégrité biologique restaurée** : les valeurs absentes ont été imputées sans fausser la distribution centrale.
- **Équité des variables** : toutes les *features* sont centrées et réduites.

Nous exportons maintenant ces matrices sous format `.csv` et le `scaler` sous format `.pkl` vers le dossier `data/processed/` et `models/`, afin de les exploiter directement dans notre troisième et dernière étape : la modélisation et l'interprétabilité SHAP (`03_modeling.ipynb`).


In [6]:
X_train_scaled.to_csv("../data/processed/X_train_clean.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test_clean.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']